用Node2Vec得到的特征来跑regression

In [28]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [29]:
# ======== Paths (按你仓库结构改一下相对路径) ========
TABULAR_PATH = Path("../../data/processed/training_level1_full.csv")  # 你现有训练表（包含 on-court + off-court + award/injury 也没事）
EMB_PATH     = Path("../embeddings/node2vec_L1A_player_embeddings.csv")

assert TABULAR_PATH.exists(), f"找不到 {TABULAR_PATH}"
assert EMB_PATH.exists(), f"找不到 {EMB_PATH}"

df = pd.read_csv(TABULAR_PATH)
emb = pd.read_csv(EMB_PATH)

print("tabular:", df.shape)
print("emb    :", emb.shape)
df.head()

tabular: (2082, 132)
emb    : (991, 66)


,player_id,season,salary_usd,log_salary,salary_cap,salary_cap_ratio,log_salary_cap_ratio,salary_cap_equiv,Age,GP,...,award_NBA Eastern Conference Finals MVP,award_NBA Western Conference Finals MVP,award_Rookie of the Year,award_Sixth Man of the Year,award_Twyman-Stokes Teammate of the Year Award,award_total,injury_events,injury_unique_dates,injury_unique_categories,injury_any
0,1,2020,43006362.0,17.576859,109140000.0,0.394048,0.332212,5.539837e+07,33,63,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,2,2020,28751775.0,17.174210,109140000.0,0.263439,0.233838,3.703642e+07,28,60,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,3,2020,31626953.0,17.269520,109140000.0,0.289783,0.254474,4.074006e+07,30,67,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0
3,4,2020,29542010.0,17.201324,109140000.0,0.270680,0.239552,3.805435e+07,27,51,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,5,2020,27528090.0,17.130718,109140000.0,0.252227,0.224924,3.546013e+07,26,61,...,0.0,0.0,0.0,0.0,0.0,4.0,2.0,2.0,1.0,True


In [30]:
ID_COLS = ["player_id", "season"]
TARGET_COL = "log_salary"   # 你主线实验用 log_salary

assert TARGET_COL in df.columns, f"{TARGET_COL} 不在 tabular 表里"
assert all(c in df.columns for c in ID_COLS), "tabular 缺少 player_id 或 season"
assert all(c in emb.columns for c in ["player_id"]), "embedding 缺少 player_id"

# 目标相关列：绝对不能进特征（防泄露）
TARGET_LEAK_BASE = [
    "salary_usd", "log_salary",
    "salary_cap", "salary_cap_ratio", "log_salary_cap_ratio",
    "salary_cap_equiv",
]
TARGET_LEAK_COLS = [c for c in TARGET_LEAK_BASE if c != TARGET_COL]

TEXT_OR_ID_COLS = [
    "Player", "player_name",
    "team_abbr", "city", "state", "region",
    "agent_name", "agent_name_all",
]

# 基础清洗
df = df.copy()

In [31]:
# 候选特征：先排除ID/target/leak/text
candidate_feature_cols = [
    c for c in df.columns
    if c not in ID_COLS + [TARGET_COL] + TARGET_LEAK_COLS + TEXT_OR_ID_COLS
]

# 只保留数值/布尔
numeric_bool_cols = df[candidate_feature_cols].select_dtypes(include=["number", "bool"]).columns.tolist()

# ====== 定义：ON-COURT = 数值特征中，排除掉所有 off-court（draft/team/agent）和 leaky（award/injury） ======
OFFCOURT_PREFIXES = ("draft_", "team_", "agent_")
LEAKY_PREFIXES    = ("award_", "injury_")

OFFCOURT_EXACT = [
    "Age", "age_now", "years_since_draft",
    "overall_pick", "round", "round_pick", "draft_year",
    "undrafted_flag",
]
OFFCOURT_CONTAINS = ("team", "market", "city", "state", "region", "pick")

def is_offcourt(col: str) -> bool:
    lc = col.lower()
    if col.startswith(OFFCOURT_PREFIXES):
        return True
    if col in OFFCOURT_EXACT:
        return True
    if any(k in lc for k in OFFCOURT_CONTAINS):
        return True
    return False

# 先排除 leaky（award/injury）
safe_cols = [c for c in numeric_bool_cols if not c.startswith(LEAKY_PREFIXES)]

# on-court：safe_cols 里那些不属于 off-court 的
oncourt_cols = sorted([c for c in safe_cols if not is_offcourt(c)])

print("Total numeric/bool:", len(numeric_bool_cols))
print("Safe (no award/injury):", len(safe_cols))
print("On-court cols:", len(oncourt_cols))

# sanity：确保 oncourt 里真的没有 award/injury
print("award in oncourt:", [c for c in oncourt_cols if c.startswith("award_")][:3])
print("injury in oncourt:", [c for c in oncourt_cols if c.startswith("injury_")][:3])

Total numeric/bool: 115
Safe (no award/injury): 94
On-court cols: 81
award in oncourt: []
injury in oncourt: []


In [32]:
def prepare_splits_with_emb(df_tabular: pd.DataFrame, emb_df: pd.DataFrame,
                            oncourt_cols: list, target_col: str = "log_salary"):
    d = df_tabular.copy()

    # 强制类型一致，避免 merge 静默失败
    d["player_id"] = d["player_id"].astype(str)
    emb_df = emb_df.copy()
    emb_df["player_id"] = emb_df["player_id"].astype(str)

    # 检查 embedding 是否每个 player 唯一
    assert emb_df["player_id"].duplicated().sum() == 0, "embedding 里 player_id 有重复"

    # merge
    d2 = d.merge(emb_df, on="player_id", how="left")

    # 更稳：只抓 e0-e* 这种列，避免误抓
    emb_cols = [c for c in d2.columns if c.startswith("e") and c[1:].isdigit()]
    assert len(emb_cols) > 0, "没抓到 embedding 列（e0,e1,...），检查 embedding 文件列名"

    # 关键 sanity：missing rate
    missing_rate = d2[emb_cols].isna().any(axis=1).mean()
    print(f"[sanity] embedding missing row rate = {missing_rate:.4f}")

    # 如果缺失太多，直接报错（否则你填0会把模型稀释到看不出来）
    if missing_rate > 0.01:
        raise ValueError(
            f"embedding 缺失率 {missing_rate:.2%} 太高，说明 merge 没对上或图不覆盖。先修复再跑。"
        )

    d2[emb_cols] = d2[emb_cols].fillna(0.0)

    train_df = d2[d2["season"] < 2024].copy().fillna(0)
    test_df  = d2[d2["season"] == 2024].copy().fillna(0)

    y_train = train_df[target_col].values
    y_test  = test_df[target_col].values

    X_train_L0 = train_df[oncourt_cols].values
    X_test_L0  = test_df[oncourt_cols].values

    X_train_L1A = train_df[oncourt_cols + emb_cols].values
    X_test_L1A  = test_df[oncourt_cols + emb_cols].values

    return X_train_L0, X_test_L0, X_train_L1A, X_test_L1A, y_train, y_test, emb_cols


In [33]:
# 只 merge player_id（embedding 不应该带 season）
assert emb["player_id"].duplicated().sum() == 0, "embedding 里 player_id 有重复，先修复"

X_train_L0, X_test_L0, X_train_L1A, X_test_L1A, y_train, y_test, emb_cols = \
    prepare_splits_with_emb(df, emb, oncourt_cols, TARGET_COL)

print("X_train_L0 :", X_train_L0.shape)
print("X_train_L1A:", X_train_L1A.shape)
print("embedding dims:", len(emb_cols))



[sanity] embedding missing row rate = 0.0000
X_train_L0 : (1659, 81)
X_train_L1A: (1659, 145)
embedding dims: 64


In [34]:
def eval_reg(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2


In [ ]:
def run_two_models(level_name, Xtr, Xte, ytr, yte, seed = 42):
    rows = []

    # Ridge
    ridge = make_pipeline(
        StandardScaler(),
        Ridge(alpha=10.0)
    )
    ridge.fit(Xtr, ytr)
    pred = ridge.predict(Xte)
    mae, rmse, r2 = eval_reg(yte, pred)
    rows.append({"level": level_name, "model": "Ridge", "mae": mae, "rmse": rmse, "r2": r2})

    # RandomForest (mild regularization)
    rf = RandomForestRegressor(
        n_estimators=500,
        max_depth=20,
        min_samples_leaf=5,
        min_samples_split=10,
        max_features="sqrt",
        n_jobs=-1,
        random_state=seed,
    )
    rf.fit(Xtr, ytr)
    pred = rf.predict(Xte)
    mae, rmse, r2 = eval_reg(yte, pred)
    rows.append({"level": level_name, "model": "RandomForest", "mae": mae, "rmse": rmse, "r2": r2})

    return rows

results = []
results += run_two_models("L0′ (On-court)", X_train_L0, X_test_L0, y_train, y_test)
results += run_two_models("L1-A (On-court + Node2Vec)", X_train_L1A, X_test_L1A, y_train, y_test)

paper_table = pd.DataFrame(results).sort_values(["level", "model"]).reset_index(drop=True)
paper_table

,level,model,mae,rmse,r2
0,L0′ (On-court),RandomForest,0.697111,0.858989,0.453288
1,L0′ (On-court),Ridge,0.698760,0.871353,0.437437
2,L1-A (On-court + Node2Vec),RandomForest,0.668277,0.823187,0.497911
3,L1-A (On-court + Node2Vec),Ridge,0.689669,0.863167,0.447957


In [36]:
def get_r2(level, model):
    return float(paper_table.query("level == @level and model == @model")["r2"].values[0])

delta_rf = get_r2("L1-A (On-court + Node2Vec)", "RandomForest") - get_r2("L0′ (On-court)", "RandomForest")
delta_ridge = get_r2("L1-A (On-court + Node2Vec)", "Ridge") - get_r2("L0′ (On-court)", "Ridge")

print(f"ΔR² (RF,   L1-A − L0′): {delta_rf:.3f}")
print(f"ΔR² (Ridge,L1-A − L0′): {delta_ridge:.3f}")

paper_table


ΔR² (RF,   L1-A − L0′): 0.045
ΔR² (Ridge,L1-A − L0′): 0.011


,level,model,mae,rmse,r2
0,L0′ (On-court),RandomForest,0.697111,0.858989,0.453288
1,L0′ (On-court),Ridge,0.698760,0.871353,0.437437
2,L1-A (On-court + Node2Vec),RandomForest,0.668277,0.823187,0.497911
3,L1-A (On-court + Node2Vec),Ridge,0.689669,0.863167,0.447957


In [37]:
# 输出成csv给paper用
OUT_DIR = Path("../results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / "paper_table_L0prime_vs_L1A_node2vec.csv"
paper_table.to_csv(out_path, index=False)
print("saved:", out_path)

saved: ..\results\paper_table_L0prime_vs_L1A_node2vec.csv


In [38]:
def eval_one_embedding_file(emb_path: Path):
    emb_df = pd.read_csv(emb_path)
    X_train_L0, X_test_L0, X_train_L1A, X_test_L1A, y_train, y_test, emb_cols = \
        prepare_splits_with_emb(df, emb_df, oncourt_cols, TARGET_COL)

    results = []
    results += run_two_models("L0′ (On-court)", X_train_L0, X_test_L0, y_train, y_test)
    results += run_two_models(f"L1-A (On-court + Node2Vec) [{emb_path.stem}]", X_train_L1A, X_test_L1A, y_train, y_test)

    out = pd.DataFrame(results).sort_values(["level", "model"]).reset_index(drop=True)

    def get_r2(level, model):
        return float(out.query("level == @level and model == @model")["r2"].values[0])

    delta_rf = get_r2(f"L1-A (On-court + Node2Vec) [{emb_path.stem}]", "RandomForest") - get_r2("L0′ (On-court)", "RandomForest")
    delta_ridge = get_r2(f"L1-A (On-court + Node2Vec) [{emb_path.stem}]", "Ridge") - get_r2("L0′ (On-court)", "Ridge")

    return out, delta_rf, delta_ridge


In [39]:
# 单个调用

out, d_rf, d_ridge = eval_one_embedding_file(Path("../embeddings/node2vec_L1A_player_embeddings.csv"))
out

[sanity] embedding missing row rate = 0.0000


,level,model,mae,rmse,r2
0,L0′ (On-court),RandomForest,0.697111,0.858989,0.453288
1,L0′ (On-court),Ridge,0.698760,0.871353,0.437437
2,L1-A (On-court + Node2Vec) [node2vec_L1A_playe...,RandomForest,0.668277,0.823187,0.497911
3,L1-A (On-court + Node2Vec) [node2vec_L1A_playe...,Ridge,0.689669,0.863167,0.447957


In [ ]:
# 如果要尝试多个p 和q的组合
GRID_DIR = Path("../embeddings/node2vec_grid")
emb_files = sorted(GRID_DIR.glob("*.csv"))
print("found:", len(emb_files))

for f in emb_files[:3]:
    out, d_rf, d_ridge = eval_one_embedding_file(f)
    print(f.name, d_rf, d_ridge)

found: 0
